In [1]:
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits
from pathlib import Path
from IPython.display import display

# Load dataset pertama, hanya kolom yang diperlukan.
features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
data_path = Path('data/archive/songs_with_attributes_and_lyrics.csv')
df = pd.read_csv(data_path, usecols=['id', 'name', 'artists'] + features)
df = df.replace({np.inf: np.nan, -np.inf: np.nan})
jumlah_awal = len(df)
df = df.dropna(subset=features).reset_index(drop=True)
X = df[features].values

# Standardisasi fitur
scaler = StandardScaler()
X_scaled = np.ascontiguousarray(scaler.fit_transform(X), dtype=np.float32)
k = 10
faiss.omp_set_num_threads(4)
print('Jumlah lagu:', len(df))
print('Jumlah fitur:', X_scaled.shape[1])
print('Baris fitur kosong yang dihapus:', jumlah_awal - len(df))
hasil_waktu = []

Jumlah lagu: 955320
Jumlah fitur: 9
Baris fitur kosong yang dihapus: 0


In [2]:
# Exact Nearest Neighbor (brute-force)
# Query dibagi batch agar penggunaan memori lebih ringan; semua lagu tetap dicari.
start = time.perf_counter()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
idx_exact = np.empty((len(X_scaled), k), dtype=np.int64)
dist_exact = np.empty((len(X_scaled), k), dtype=np.float32)
batch_size = 10000
with threadpool_limits(limits=4):
    for awal in range(0, len(X_scaled), batch_size):
        akhir = min(awal + batch_size, len(X_scaled))
        dist_exact[awal:akhir], idx_exact[awal:akhir] = nn.kneighbors(X_scaled[awal:akhir])
time_exact = time.perf_counter() - start
print(f'Exact NN done in {time_exact:.3f} s')
hasil_waktu.append({'Model': 'Exact NN', 'Waktu total (detik)': time_exact})

Exact NN done in 989.915 s


In [3]:
# Annoy
start = time.perf_counter()
f = X_scaled.shape[1]
index_annoy = AnnoyIndex(f, 'euclidean')
index_annoy.set_seed(42)
for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)
index_annoy.build(10, n_jobs=4)
idx_annoy = np.asarray([index_annoy.get_nns_by_vector(v, k) for v in X_scaled], dtype=np.int64)
time_annoy = time.perf_counter() - start
print(f'Annoy done in {time_annoy:.3f} s')
hasil_waktu.append({'Model': 'Annoy', 'Waktu total (detik)': time_annoy})
del index_annoy

Annoy done in 39.706 s


In [4]:
# HNSW
start = time.perf_counter()
p_hnsw = hnswlib.Index(space='l2', dim=X_scaled.shape[1])
p_hnsw.init_index(max_elements=len(X_scaled), ef_construction=200, M=16, random_seed=42)
p_hnsw.set_num_threads(4)
p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(200)
idx_hnsw, dist_hnsw = p_hnsw.knn_query(X_scaled, k=k)
time_hnsw = time.perf_counter() - start
print(f'HNSW done in {time_hnsw:.3f} s')
hasil_waktu.append({'Model': 'HNSW', 'Waktu total (detik)': time_hnsw})
del p_hnsw

HNSW done in 87.756 s


In [5]:
# FAISS IVF
start = time.perf_counter()
quantizer = faiss.IndexFlatL2(X_scaled.shape[1])
index_faiss = faiss.IndexIVFFlat(quantizer, X_scaled.shape[1], 100, faiss.METRIC_L2)
index_faiss.cp.seed = 42
index_faiss.train(X_scaled)
index_faiss.add(X_scaled)
index_faiss.nprobe = 10
dist_faiss, idx_faiss = index_faiss.search(X_scaled, k)
time_faiss = time.perf_counter() - start
print(f'FAISS IVF done in {time_faiss:.3f} s')
hasil_waktu.append({'Model': 'FAISS IVF', 'Waktu total (detik)': time_faiss})
del index_faiss, quantizer

FAISS IVF done in 147.202 s


In [6]:
# Top-5 neighbors dari lagu pertama, sesuai modul (termasuk lagu itu sendiri).
print('Lagu pertama:', df.loc[0, 'name'], '-', df.loc[0, 'artists'])
print('Exact NN:', idx_exact[0][:5])
print('Annoy:', idx_annoy[0][:5])
print('HNSW:', idx_hnsw[0][:5])
print('FAISS:', idx_faiss[0][:5])

def recall_at_k(I_pred, I_gt, k):
    correct = 0
    for pred, gt in zip(I_pred, I_gt):
        correct += len(set(pred[:k]) & set(gt[:k]))
    return correct / (len(I_gt) * k)

tabel_hasil = pd.DataFrame(hasil_waktu)
tabel_hasil['Recall@10'] = [1.0, recall_at_k(idx_annoy, idx_exact, k),
                          recall_at_k(idx_hnsw, idx_exact, k),
                          recall_at_k(idx_faiss, idx_exact, k)]
display(tabel_hasil)

# Nama track tetangga agar hasil rekomendasi mudah dibaca.
hasil_track = []
for model, indices in [('Exact NN', idx_exact), ('Annoy', idx_annoy),
                       ('HNSW', idx_hnsw), ('FAISS IVF', idx_faiss)]:
    for urutan, posisi in enumerate(indices[0][:5], start=1):
        lagu = df.iloc[int(posisi)]
        hasil_track.append({'Model': model, 'Urutan': urutan, 'Index': int(posisi),
                            'Nama lagu': lagu['name'], 'Artis': lagu['artists']})
with pd.option_context('display.max_colwidth', None):
    display(pd.DataFrame(hasil_track))

Lagu pertama: ! - HELLYEAH
Exact NN: [     0 394553 764272 837727 749223]
Annoy: [     0 764272 837727  61511 625227]
HNSW: [     0 394553 764272 837727 749223]
FAISS: [     0 394553 764272 837727 749223]


,Model,Waktu total (detik),Recall@10
0,Exact NN,989.914903,1.000000
1,Annoy,39.705909,0.835607
2,HNSW,87.755774,0.994491
3,FAISS IVF,147.202024,0.998141


,Model,Urutan,Index,Nama lagu,Artis
0,Exact NN,1,0,!,HELLYEAH
1,Exact NN,2,394553,Intro (Who The Fuck Are You'),C-Lekktor
2,Exact NN,3,764272,Testimonium,Lacrimosa
3,Exact NN,4,837727,Ultimate Satisfaction - The Field Remix,Bear In Heaven
4,Exact NN,5,749223,Sæglópur,Sigur Rós
5,Annoy,1,0,!,HELLYEAH
6,Annoy,2,764272,Testimonium,Lacrimosa
7,Annoy,3,837727,Ultimate Satisfaction - The Field Remix,Bear In Heaven
8,Annoy,4,61511,Autumn Interlude,Amethystium
9,Annoy,5,625227,Púrpura,Margaritas Podridas


StandardScaler menyamakan skala fitur agar tempo dan loudness tidak mendominasi jarak. Exact NN menjadi pembanding, sedangkan Annoy, HNSW, dan FAISS IVF mencari tetangga dengan aproksimasi. Recall mendekati 1 berarti hasil semakin mirip Exact NN. Waktu total mencakup persiapan index dan pencarian semua lagu, sesuai kode modul. Lagu query tetap ikut dalam hasil; beberapa track juga bisa memiliki fitur identik.